In [5]:
#Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
#Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [19]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content" : text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content" : text}
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    
    return message.content[0].text

In [9]:
# Make a starting list of messages
messages = []

add_user_message(
    messages,
    "write a 1 sentence descriptio of a fake database",
    )

stream = client.messages.create(
    model=model,
    max_tokens=300,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)
    


RawMessageStartEvent(message=Message(id='msg_016DSJ2JVcTNzodoQGzuSfD1', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Fake Database Description\n\nThe "CloudVault" database is a distributed NoSQL system designed to store and retrieve', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaE

In [17]:
messages = []

add_user_message(
    messages,
    "write a 1 sentence descriptio of a fake database",
    )

with client.messages.stream(
    model = model,
    max_tokens = 300,
    messages = messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

stream.get_final_message()

# Fake Database Description

The "CloudVault™ Personal Archive" is a cloud-based storage system that automatically organizes and backs up all your digital memories, files, and documents while providing real-time synchronization across unlimited devices and AI-powered search capabilities.

ParsedMessage(id='msg_01QABYMTyD3jGVx9ipRdif2V', container=None, content=[ParsedTextBlock(citations=None, text='# Fake Database Description\n\nThe "CloudVault™ Personal Archive" is a cloud-based storage system that automatically organizes and backs up all your digital memories, files, and documents while providing real-time synchronization across unlimited devices and AI-powered search capabilities.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=19, output_tokens=59, server_tool_use=None, service_tier='standard'))

In [20]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "Name": "MySimpleRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [21]:
import json

json.loads(text.strip())

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)
text = chat(messages)
text.strip()


'# Three AWS CLI Commands\n\n1. **List all S3 buckets:**\n```bash\naws s3 ls\n```\n\n2. **Describe all EC2 instances:**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID:**\n```bash\naws sts get-caller-identity\n```'

In [40]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls
aws ec2 describe-instances --region us-east-1
aws dynamodb list-tables


In [ ]:
messages = []

prompt = """
Give me exactly 3 bash commands to filter names from a list, one per line, no explanations
"""
add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])

print(text)


grep "^[A-Z]" names.txt
awk '{print $1}' names.txt
sed -n '/^[a-z]/p' names.txt

